# 03: Flow Decomposition & Kinematics

Use `KinematicsAnalyzer` to decompose a surface flow field:
Helmholtz-Hodge (irrotational / solenoidal), the surface **metric tensor** (first
fundamental form), and the cumulative **Lagrangian** strain over a timeseries.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from seamless import FlowEstimator, KinematicsAnalyzer
from seamless.config import ThreeDNativeConfig, KinematicsConfig

## Load the timeseries and estimate flow (PIV)

PIV is fast (no training) and gives a 3D velocity field on every frame pair.

In [ ]:
estimator = FlowEstimator.from_projection_h5(
    Path('data/synthetic/ellipsoid_projection.h5'),
    source_file=Path('data/synthetic/ellipsoid.h5'),
)
piv_fields = estimator.estimate('piv')
print(f'{len(piv_fields)} flow fields, uv_res={estimator.frames[0].uv_res}')
analyzer = KinematicsAnalyzer(estimator.frames, piv_fields,
                              config=KinematicsConfig(hhd_epochs=0))

## Helmholtz-Hodge decomposition

The discrete (FFT) HHD splits the tangential velocity into an irrotational (curl-free)
and a solenoidal (divergence-free) part. _(The continuous harmonic residual via
`extract_harmonic_component` is meant for downsampled point clouds, so we disable it
here with `hhd_epochs=0`.)_

In [ ]:
hhd = analyzer.decompose_hhd(piv_fields[0])
print('keys:', sorted(hhd.keys()))

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(hhd['divergence'], cmap='RdBu_r'); axes[0].set_title('divergence')
axes[1].imshow(np.linalg.norm(hhd['v_irrotational'], axis=-1), cmap='magma')
axes[1].set_title('|v_irrotational|')
axes[2].imshow(np.linalg.norm(hhd['v_solenoidal'], axis=-1), cmap='magma')
axes[2].set_title('|v_solenoidal|')
for a in axes: a.axis('off')
plt.tight_layout(); plt.show()

## Surface metric tensor (first fundamental form)

`g = [[E, F], [F, G]]` at every UV pixel, from the parametric position map.

In [ ]:
g = analyzer.compute_metric_tensor(estimator.frames[0])   # (H, W, 2, 2)
print('metric tensor:', g.shape)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (i, j, name) in zip(axes, [(0,0,'E'), (0,1,'F'), (1,1,'G')]):
    im = ax.imshow(g[..., i, j], cmap='viridis'); ax.set_title(name); ax.axis('off')
    fig.colorbar(im, ax=ax)
plt.tight_layout(); plt.show()

## Cumulative Lagrangian metrics

Lagrangian accumulation needs an MLP-based flow (to advect material points), so we
estimate 3D-native flow over a short sub-sequence. The deformation gradient is
integrated analytically; the result is `{log_J, areal_change, strain}` per timepoint.

_This cell trains one FlowMLP per pair and integrates over the dense grid — it is the
compute-heavy cell; reduce `num_iters` / the frame count for a faster run._

In [ ]:
# Short sub-sequence + reduced iterations for the demo.
estimator.frames = estimator.frames[:4]
estimator.three_d_config = ThreeDNativeConfig(num_iters=50)
flow3d = estimator.estimate('3d_native')

lagr_analyzer = KinematicsAnalyzer(estimator.frames, flow3d,
                                   config=KinematicsConfig(lagrangian_grid_res=64))
lag = lagr_analyzer.compute_lagrangian()
last_t = max(lag)
m = lag[last_t]['analytical']
print('lagrangian timepoints:', sorted(lag), '| showing t =', last_t)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (key, cmap) in zip(axes, [('log_J','RdBu_r'), ('areal_change','PRGn'), ('strain','viridis')]):
    im = ax.imshow(m[key], cmap=cmap); ax.set_title(f'{key} (t={last_t})'); ax.axis('off')
    fig.colorbar(im, ax=ax)
plt.tight_layout(); plt.show()

## Summary

`KinematicsAnalyzer` provides the full decomposition toolkit on top of any `FlowField`:
Helmholtz-Hodge split, the surface metric tensor, and cumulative Lagrangian strain
(`log_J`, `areal_change`, `strain`) integrated over a timeseries.